# Phase 6 — Leakage-Safe Feature Engineering

## Goal

Convert the validated crop-yield table into model-ready features without allowing validation or test information to influence preprocessing.


## Setup

Run this notebook from the repository root after completing the data-validation workflow. The preprocessor is fitted only on observations from 1990–2007.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from crop_yield.preprocessing import (
    build_preprocessor,
    split_features_target,
)
from crop_yield.splitting import temporal_train_validation_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Steps

### 1. Load the validated modelling table


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)
print(f"Rows: {len(crop_yield):,}")
crop_yield.head()


### 2. Split through time before preprocessing

Training covers 1990–2007, validation covers 2008–2010, and test covers 2011–2013. This order mirrors the intended task of predicting future yields from past data.


In [ ]:
temporal_split = temporal_train_validation_test_split(crop_yield)

split_summary = pd.DataFrame(
    [
        {
            "partition": name,
            "rows": len(partition),
            "first_year": partition["year"].min(),
            "last_year": partition["year"].max(),
            "countries": partition["area"].nunique(),
            "crops": partition["item"].nunique(),
        }
        for name, partition in [
            ("train", temporal_split.train),
            ("validation", temporal_split.validation),
            ("test", temporal_split.test),
        ]
    ]
)
split_summary


### 3. Separate features from the prediction target

`yield_hg_per_ha` is the target. It must not appear among the input features.


In [ ]:
X_train, y_train = split_features_target(temporal_split.train)
X_validation, y_validation = split_features_target(
    temporal_split.validation
)
X_test, y_test = split_features_target(temporal_split.test)

print(f"Training features: {X_train.shape}")
print(f"Training target: {y_train.shape}")
X_train.head()


### 4. Fit preprocessing on training only

- Country and crop are one-hot encoded without inventing an order.
- Unknown categories are ignored so future countries such as Sudan do not cause failure.
- Year, rainfall, and temperature are standardized.
- Pesticide totals are transformed with `log1p` before standardization to reduce right-skew.


In [ ]:
preprocessor = build_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

processed_shapes = pd.DataFrame(
    {
        "rows": [
            len(X_train_processed),
            len(X_validation_processed),
            len(X_test_processed),
        ],
        "features": [
            X_train_processed.shape[1],
            X_validation_processed.shape[1],
            X_test_processed.shape[1],
        ],
    },
    index=["train", "validation", "test"],
)
processed_shapes


### 5. Inspect the generated feature names

The fitted training preprocessor creates separate country and crop indicators plus four numeric features.


In [ ]:
feature_names = preprocessor.get_feature_names_out().tolist()

pd.Series({
    "total_model_features": len(feature_names),
    "country_indicators": sum(name.startswith("area_") for name in feature_names),
    "crop_indicators": sum(name.startswith("item_") for name in feature_names),
    "numeric_features": sum(
        name in {
            "year",
            "average_rainfall_mm_per_year",
            "average_temperature_c",
            "pesticides_tonnes",
        }
        for name in feature_names
    ),
})


### 6. Verify unknown-category behaviour

Sudan appears in test but not training. The encoder should transform the row successfully without creating a feature learned from future data.


In [ ]:
training_countries = set(X_train["area"])
test_countries = set(X_test["area"])
unseen_test_countries = sorted(test_countries - training_countries)

print(f"Unseen test countries: {unseen_test_countries}")
print(f"Sudan feature learned during training: {'area_Sudan' in feature_names}")


### 7. Demonstrate the pesticide transformation

`log1p` compresses the distance between very large pesticide totals while preserving their order.


In [ ]:
pesticide_example = pd.DataFrame(
    {
        "original_tonnes": [0.0, 10.0, 1_000.0, 100_000.0],
    }
)
pesticide_example["log1p_value"] = np.log1p(
    pesticide_example["original_tonnes"]
)
pesticide_example


## Checks


In [ ]:
assert len(X_train_processed) == 9_609
assert len(X_validation_processed) == 1_751
assert len(X_test_processed) == 1_770
assert X_train_processed.shape[1] == 114
assert X_validation_processed.shape[1] == 114
assert X_test_processed.shape[1] == 114
assert "yield_hg_per_ha" not in X_train.columns
assert unseen_test_countries == ["Sudan"]
assert "area_Sudan" not in feature_names
assert np.isfinite(X_train_processed.to_numpy()).all()
assert np.isfinite(X_validation_processed.to_numpy()).all()
assert np.isfinite(X_test_processed.to_numpy()).all()
print("All Phase 6 feature-engineering checks passed.")


## Next Steps

- Combine this preprocessor with a naive benchmark and a linear-regression baseline.
- Fit every complete pipeline on training only.
- Use validation metrics to compare models.
- Keep the 2011–2013 test set untouched until the final approach is selected.

**Interpretation note:** transforming a feature does not make its association causal. Pesticide totals remain national-level measurements with important confounding limitations.
